In [ ]:
import torch
import sqlite3
import os
import re
import json
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig, get_cosine_schedule_with_warmup
from peft import LoraConfig, get_peft_model, PeftModel
from datasets import load_dataset
from torch.optim import AdamW
from tqdm.auto import tqdm
from bitsandbytes.optim import PagedAdamW32bit

In [ ]:
MODEL_PATH       = "/mnt/storage_C1/igorzwirtes/poster_ic/qwen2.5coder"
ADAPTER_PATH     = "/mnt/storage_C1/igorzwirtes/poster_ic/lora_weights/r64q4a128_2"
SPIDER_DB_DIR    = "/mnt/storage_C1/igorzwirtes/poster_ic/spider/database"
SPIDER_TABLES    = "/mnt/storage_C1/igorzwirtes/poster_ic/spider/tables.json"
OUTPUT_DIR       = "/mnt/storage_C1/igorzwirtes/poster_ic/lora_weights/direct"
USE_BF16         = torch.cuda.is_available() and torch.cuda.is_bf16_supported()
USE_FP16         = torch.cuda.is_available() and not USE_BF16
USE_CPU          = not torch.cuda.is_available()

LR               = 1e-5
NUM_EPOCHS       = 1
GRAD_ACCUM       = 8
WARMUP_STEPS     = 20
MAX_PROMPT_LEN   = 605
MAX_NEW_TOKENS   = 96
NUM_SAMPLES      = 16       
LOGGING_STEPS    = 1
EVAL_STEPS       = 200
SAVE_STEPS       = 200
SAVE_TOTAL_LIMIT = 3

In [ ]:
with open(SPIDER_TABLES) as f:
    tables_data = json.load(f)

def format_schema_with_fk(db):
    col_names = db["column_names_original"]
    lines = []
    for i, table in enumerate(db["table_names_original"]):
        cols = [col[1] for col in col_names if col[0] == i]
        lines.append(f"  {table}({', '.join(cols)})")
    if db.get("foreign_keys"):
        lines.append("  Foreign keys:")
        for fk in db["foreign_keys"]:
            c1 = col_names[fk[0]]
            c2 = col_names[fk[1]]
            t1 = db["table_names_original"][c1[0]]
            t2 = db["table_names_original"][c2[0]]
            lines.append(f"    {t1}.{c1[1]} → {t2}.{c2[1]}")
    return "\n".join(lines)

schema_index = {db["db_id"]: format_schema_with_fk(db) for db in tables_data}

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)
tokenizer.pad_token    = tokenizer.eos_token
tokenizer.pad_token_id = tokenizer.eos_token_id

In [ ]:
dataset    = load_dataset("spider")
train_data = dataset["train"]
val_data   = dataset["validation"]

FEW_SHOT = """
Example 1:
Database: concert_singer
Schema:
  singer(Singer_ID, Name, Country, Age)
Question: How many singers are from USA?
SQL: SELECT COUNT(*) FROM singer WHERE Country = 'USA';

Example 2:
Database: concert_singer  
Schema:
  concert(concert_ID, Name, Stadium_ID)
  stadium(Stadium_ID, Name, Capacity)
Question: What are the names of all stadiums?
SQL: SELECT Name FROM stadium;
"""

def build_prompt(example):
    schema = schema_index.get(example["db_id"], "")
    messages = [
        {"role": "system", "content": (
            "You are a Text-to-SQL translator.\n"
            "Generate a valid SQLite SQL query.\n"
            "Use only tables and columns from the schema.\n"
            "Output only the SQL query.\n"
            "Do not explain.\n"
            "Do not use markdown."
        )},
        {
            "role": "user",
            "content": (
                f"Database: {example['db_id']}\n"
                f"Schema:\n{schema}\n\n"
                f"Question: {example['question']}"
            )
        }
    ]
    return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

# Testar uso máximo de VRAM
#train_data = sorted(train_data, key=lambda ex: len(tokenizer(build_prompt(ex))["input_ids"]), reverse=True)

In [ ]:
def extract_sql(text):
    # bloco markdown
    match = re.search(
        r"```(?:sql)?\s*(.*?)```",
        text,
        re.DOTALL | re.IGNORECASE
    )

    if match:
        sql = match.group(1).strip()
    else:
        # pega primeira query SQL
        match = re.search(
            r"(SELECT|INSERT|UPDATE|DELETE|WITH)\b.*?;",
            text,
            re.DOTALL | re.IGNORECASE
        )

        if match:
            sql = match.group(0).strip()
        else:
            sql = text.strip()

    # remove comentários
    sql = re.sub(r"--.*", "", sql)

    # remove markdown sobrando
    sql = sql.replace("```", "").strip()

    return sql

def execution_reward(db_id, pred_sql, gold_sql):
    db_path = os.path.join(SPIDER_DB_DIR, db_id, f"{db_id}.sqlite")

    conn = None

    try:
        conn = sqlite3.connect(db_path)
        cur = conn.cursor()

        cur.execute(pred_sql)
        pred_res = sorted(cur.fetchall())

        cur.execute(gold_sql)
        gold_res = sorted(cur.fetchall())

        return float(pred_res == gold_res)

    except Exception as e:
        print("SQL ERROR:", e)
        print(pred_sql)
        return 0.0

    finally:
        if conn is not None:
            conn.close()

In [ ]:
compute_dtype = torch.bfloat16 if USE_BF16 else torch.float16

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=compute_dtype,
    bnb_4bit_use_double_quant=True,
)

base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_PATH,
    quantization_config=bnb_config,
    device_map="auto",
)
base_model.config.use_cache = False

base_model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
'''
lora_config = LoraConfig(
    r=64,
    lora_alpha=128,
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj"      
    ],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)
'''
#model = get_peft_model(base_model, lora_config)

model = PeftModel.from_pretrained(
    base_model,
    ADAPTER_PATH,
    is_trainable=True,
)

model.print_trainable_parameters()

optimizer = PagedAdamW32bit(model.parameters(), lr=LR)

total_steps = (len(train_data) * NUM_EPOCHS) // GRAD_ACCUM
scheduler   = get_cosine_schedule_with_warmup(optimizer, num_warmup_steps=WARMUP_STEPS, num_training_steps=total_steps)

In [ ]:
def evaluate(model, val_data, max_examples=200):
    model.eval()
    correct = 0
    total   = min(max_examples, len(val_data))
    device  = next(model.parameters()).device
    for example in list(val_data)[:total]:
        prompt = build_prompt(example)
        inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=MAX_PROMPT_LEN).to(device)
        with torch.no_grad():
            output = model.generate(
                **inputs,
                max_new_tokens=MAX_NEW_TOKENS,
                do_sample=False,
                pad_token_id=tokenizer.eos_token_id,
            )
        gen_tokens = output[0][inputs["input_ids"].shape[1]:]
        pred_sql   = extract_sql(tokenizer.decode(gen_tokens, skip_special_tokens=True))
        correct   += execution_reward(example["db_id"], pred_sql, example["query"])
    model.train()
    return correct / total

In [ ]:
device         = next(model.parameters()).device
global_step    = 0
best_eval      = 0.0
saved_checkpoints = []

for epoch in range(NUM_EPOCHS):
    model.train()
    optimizer.zero_grad()
    total_loss   = 0.0
    total_reward = 0.0
    window_loss   = 0.0
    window_reward = 0.0
    window_count  = 0
    step_loss_accum   = 0.0
    step_reward_accum = 0.0

    for i, example in enumerate(tqdm(train_data, desc=f"Epoch {epoch+1}", disable=False)):
        prompt = build_prompt(example)
        inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=MAX_PROMPT_LEN).to(device)

        rewards        = []
        log_probs_list = []
        entropies = []

        for _ in range(NUM_SAMPLES):
            model.eval()
            with torch.no_grad():
                output = model.generate(
                    **inputs,
                    max_new_tokens=MAX_NEW_TOKENS,
                    do_sample=True,
                    temperature=0.3,
                    top_p=0.9,
                    pad_token_id=tokenizer.eos_token_id,
                    eos_token_id=tokenizer.eos_token_id,
                )
            model.train()

            gen_tokens = output[0][inputs["input_ids"].shape[1]:]
            pred_sql   = extract_sql(tokenizer.decode(gen_tokens, skip_special_tokens=True))
            pred_sql = pred_sql.split(";")[0] + ";"
            reward     = execution_reward(example["db_id"], pred_sql, example["query"])
            rewards.append(reward)
            
            logits = model(input_ids=output).logits
            shift_logits = logits[0, inputs["input_ids"].shape[1]-1:-1]
            shift_labels = gen_tokens

            log_probs = torch.nn.functional.log_softmax(shift_logits, dim=-1)

            token_log_probs = log_probs[
                torch.arange(len(shift_labels), device=device),
                shift_labels
            ]

            seq_log_prob = token_log_probs.mean()

            log_probs_list.append(seq_log_prob)

            entropy = -(log_probs.exp() * log_probs).sum(dim=-1).mean()
            entropies.append(entropy)

        # REINFORCE com baseline
        rewards_tensor = torch.tensor(rewards, dtype=torch.float32)

        advantages = rewards_tensor - rewards_tensor.mean()
        advantages = advantages / (advantages.std() + 1e-8)
        advantages = advantages.detach()

        mean_entropy = torch.stack(entropies).mean()

        policy_loss = torch.stack([
            -adv.to(device) * lp
            for adv, lp in zip(advantages, log_probs_list)
        ]).sum() / NUM_SAMPLES

        total_loss_step = policy_loss - 0.001 * mean_entropy

        (total_loss_step / GRAD_ACCUM).backward()

        step_loss_accum   += total_loss_step.detach().item()
        step_reward_accum += rewards_tensor.mean().detach().item()
        total_reward      += rewards_tensor.mean().detach().item()

        if (i + 1) % GRAD_ACCUM == 0:
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            scheduler.step()
            optimizer.zero_grad()
            global_step += 1

            # Agora window pega a média real do batch
            window_loss   += step_loss_accum / GRAD_ACCUM
            window_reward += step_reward_accum / GRAD_ACCUM
            window_count  += 1

            # Reseta acumuladores do batch
            step_loss_accum   = 0.0
            step_reward_accum = 0.0

            if global_step % LOGGING_STEPS == 0:
                print(f"Step {global_step} | loss: {window_loss/window_count:.4f} | reward: {window_reward/window_count:.4f} | lr: {scheduler.get_last_lr()[0]:.2e}")
                window_loss   = 0.0
                window_reward = 0.0
                window_count  = 0

            if global_step % EVAL_STEPS == 0:
                ex = evaluate(model, val_data)
                avg_reward = total_reward / (i + 1)  
                print(f"Step {global_step} | Train reward: {avg_reward:.4f} | Val EX: {ex:.4f}")

            if global_step % SAVE_STEPS == 0:
                save_path = f"{OUTPUT_DIR}/checkpoint-{global_step}"
                model.save_pretrained(save_path)
                saved_checkpoints.append(save_path)
                if len(saved_checkpoints) > SAVE_TOTAL_LIMIT:
                    import shutil
                    shutil.rmtree(saved_checkpoints.pop(0))
    if len(train_data) % GRAD_ACCUM != 0:
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()
        optimizer.zero_grad()
        global_step += 1

model.save_pretrained(f"{OUTPUT_DIR}/final")
print("Treino concluído.")